# 04. Preparación de datos: variables, etiquetas y partición

Vamos a transformar los cinco CSV originales en **tres conjuntos finales**, cada uno con una fila por máquina y hora:

- `entrenamiento.csv`
- `validación.csv`
- `prueba.csv`

Cada fila tendrá las variables de entrada y dos respuestas: `etiqueta_binaria` y `etiqueta_multiclase`. Este notebook explica y revisa la preparación. Las funciones están documentadas en `preparar_datos.py` y se utilizan aquí; el script ejecuta esas mismas funciones con un solo comando. No existen dos implementaciones distintas.

Todavía no entrenaremos modelos. Prueba se prepara y guarda separada; no exploraremos sus etiquetas ni evaluaremos modelos sobre ella.

## 1. Preparar las herramientas y cargar las funciones

Localizamos la raíz del proyecto y añadimos `Preprocesamiento` a las carpetas donde Python busca módulos. Así podemos importar el script tanto al ejecutar desde la raíz como desde esta carpeta.

In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
from IPython.display import display

# Encontrar el proyecto sin escribir una ruta personal de la computadora.
carpeta_actual = Path.cwd()
if (carpeta_actual / "Datos").is_dir():
    ruta_repo = carpeta_actual
elif (carpeta_actual.parent / "Datos").is_dir():
    ruta_repo = carpeta_actual.parent
else:
    raise FileNotFoundError("No se encontró la carpeta Datos.")

# Permitir la importación del archivo local que contiene la preparación compartida.
ruta_codigo = str(ruta_repo / "Preprocesamiento")
if ruta_codigo not in sys.path:
    sys.path.insert(0, ruta_codigo)

from preparar_datos import (
    cargar_datos, crear_variables, preparar_conjuntos, guardar_conjuntos,
    SENSORES, COMPONENTES, ERRORES, PERIODOS, HORIZONTE,
)

pd.set_option("display.max_columns", 14)
pd.set_option("display.max_rows", 25)

## 2. Leer las cinco fuentes

La telemetría será la base por máquina y hora. Los errores se convertirán en conteos, el mantenimiento en tiempo desde reemplazos y el catálogo aportará modelo y antigüedad. Las fallas se usarán exclusivamente para crear las respuestas futuras.

Los mantenimientos de 2014 se conservan como historia. Los registros de 2016 no se utilizan: dejaremos un margen al final de diciembre para no asumir seguimiento futuro completo.

In [2]:
# La función lee los CSV y verifica que las claves necesarias no estén repetidas.
datos = cargar_datos(ruta_repo)

# Mostrar únicamente la estructura de las fuentes, sin explorar casos de prueba.
resumen_fuentes = []
for nombre, tabla in datos.items():
    resumen_fuentes.append({"fuente": nombre, "columnas": ", ".join(tabla.columns)})
display(pd.DataFrame(resumen_fuentes))

,fuente,columnas
0,telemetria,"datetime, machineID, volt, rotate, pressure, v..."
1,errores,"datetime, machineID, errorID"
2,mantenimiento,"datetime, machineID, comp"
3,fallas,"datetime, machineID, failure"
4,maquinas,"machineID, model, age"


Las fuentes se conservan separadas durante la lectura. No hacemos un cruce directo de todas sus filas: eso podría multiplicar mediciones cuando hay varios errores o componentes en una hora.

## 3. Construir las variables históricas e integrar las fuentes

| Información | Variables que vamos a conservar |
|---|---|
| Telemetría | Los cuatro sensores y sus medias y desviaciones estándar de 3 y 24 horas: 16 variables móviles. |
| Errores | Cinco conteos, uno por tipo, durante las últimas 24 horas. |
| Mantenimiento | Cuatro tiempos desde el último reemplazo, uno por componente. |
| Catálogo | Modelo y antigüedad. |

Los sensores y los errores usan ventanas **(t − L, t]**: se supone que las mediciones y eventos de `t` ya fueron observados al predecir. Para reemplazos usamos **fecha < t**, la regla conservadora acordada al revisar mantenimiento.

Las ventanas se calculan por máquina **antes de separar los conjuntos**. Esto permite que la primera hora de septiembre aproveche agosto, sin utilizar datos posteriores a esa hora. No se usan ventanas centradas, relleno desde el futuro ni estadísticas aprendidas con validación o prueba.

Pedimos ventanas completas de sensores: al comienzo faltará la media y desviación de las primeras 2 horas para la ventana de 3, y de las primeras 23 para la de 24. Conservamos esos vacíos para tratarlos después con entrenamiento. La desviación es muestral (`ddof=1`). Si no existe reemplazo anterior, el tiempo queda vacío; no se inventa un evento. La ausencia de registros de error se cuenta como cero, suponiendo que el registro de eventos esté completo.

### Qué significa cada variable y cómo leer sus valores

Cada fila describe una máquina en una hora `t`. Por ejemplo, si `t` es las 10:00, una ventana de 3 horas usa las mediciones de las **08:00, 09:00 y 10:00**. Una ventana de 24 horas termina también a las 10:00 y contiene 24 mediciones, sin incluir las 10:00 del día anterior.

#### Sensores originales: 4 variables

| Columna | Significado | Cómo se obtiene | Valores posibles |
|---|---|---|---|
| `volt` | Voltaje registrado en esa hora. | Se conserva de la telemetría. | Número decimal en la escala original. |
| `rotate` | Rotación registrada en esa hora. | Se conserva de la telemetría. | Número decimal en la escala original. |
| `pressure` | Presión registrada en esa hora. | Se conserva de la telemetría. | Número decimal en la escala original. |
| `vibration` | Vibración registrada en esa hora. | Se conserva de la telemetría. | Número decimal en la escala original. |

No fijamos límites físicos ni inventamos unidades que no están indicadas en los archivos. Estos valores se conservan sin normalizar en esta preparación.

#### Promedios y desviaciones móviles: 16 variables nuevas

La **media** resume el nivel habitual del sensor durante la ventana. La **desviación estándar** resume cuánto varió: es cero si todas las mediciones fueron iguales y aumenta cuando están más separadas del promedio.

| Sensor | Media de 3 horas | Desviación de 3 horas | Media de 24 horas | Desviación de 24 horas |
|---|---|---|---|---|
| Voltaje | `volt_media_3h` | `volt_desv_3h` | `volt_media_24h` | `volt_desv_24h` |
| Rotación | `rotate_media_3h` | `rotate_desv_3h` | `rotate_media_24h` | `rotate_desv_24h` |
| Presión | `pressure_media_3h` | `pressure_desv_3h` | `pressure_media_24h` | `pressure_desv_24h` |
| Vibración | `vibration_media_3h` | `vibration_desv_3h` | `vibration_media_24h` | `vibration_desv_24h` |

- **Cómo se calcula la media:** sumar las 3 o 24 mediciones de ese sensor de la misma máquina y dividir entre 3 o 24.
- **Cómo se calcula la desviación:** medir la separación de las mediciones respecto a su media; usamos la desviación muestral, que divide la suma de diferencias al cuadrado entre `n − 1` antes de tomar la raíz cuadrada.
- **Valores posibles:** la media es un número decimal entre el mínimo y el máximo de su ventana. La desviación es un número decimal mayor o igual que cero. Ambas conservan la escala del sensor y no están limitadas a 0–1.
- **Si falta historia:** el valor queda vacío (`NaN`); no significa cero ni que el sensor falló. Exigimos 3 o 24 mediciones completas, sin mezclar máquinas.

**Ejemplo ilustrativo:** si el voltaje de las últimas tres horas fue 100, 110 y 120, `volt_media_3h = 110` y `volt_desv_3h = 10`. Si fue 110, 110 y 110, la media sigue siendo 110, pero la desviación es 0. Son ejemplos para entender el cálculo, no mediciones tomadas de prueba.

Son **4 sensores × 2 cálculos × 2 ventanas = 16 columnas nuevas**.

#### Conteos de errores: 5 variables nuevas

| Columna | Qué cuenta en las últimas 24 horas |
|---|---|
| `error1_conteo_24h` | Registros de `error1` de esa máquina. |
| `error2_conteo_24h` | Registros de `error2` de esa máquina. |
| `error3_conteo_24h` | Registros de `error3` de esa máquina. |
| `error4_conteo_24h` | Registros de `error4` de esa máquina. |
| `error5_conteo_24h` | Registros de `error5` de esa máquina. |

Primero contamos los eventos por máquina, hora y tipo. Después sumamos las últimas 24 horas, incluido `t` y excluido `t − 24 h`. Los valores son **enteros 0, 1, 2, 3…**: cero significa que no hay registros de ese tipo en la ventana. No se impone un máximo de 24 porque puede haber varios eventos por hora. Se presupone que el registro de errores está completo; al inicio del año no disponemos de errores anteriores a enero.

**Ejemplo:** si hay dos registros de `error2` en la ventana, `error2_conteo_24h = 2`; los otros tipos se cuentan por separado. El código cuenta registros, sin asumir una gravedad diferente por su nombre.

#### Tiempo desde reemplazos: 4 variables nuevas

| Columna | Qué representa |
|---|---|
| `comp1_horas_desde_reemplazo` | Horas desde el último reemplazo conocido de comp1. |
| `comp2_horas_desde_reemplazo` | Horas desde el último reemplazo conocido de comp2. |
| `comp3_horas_desde_reemplazo` | Horas desde el último reemplazo conocido de comp3. |
| `comp4_horas_desde_reemplazo` | Horas desde el último reemplazo conocido de comp4. |

Para cada máquina y componente, buscamos el reemplazo más reciente **estrictamente anterior a `t`** y calculamos:

**horas desde reemplazo = hora actual − fecha del último reemplazo**, expresado en horas.

Los valores son números no negativos; con estos registros horarios y la regla estricta, el mínimo conocido es **1 hora**. Pueden ser mucho mayores que 24. Si no hay reemplazo anterior conocido, queda `NaN`. También se usan reemplazos de 2014 cuando corresponden a historia anterior.

**Ejemplo:** si el reemplazo fue a las 06:00, a las 08:00 el valor es 2. A las propias 06:00 todavía se consulta el reemplazo anterior: no usamos un evento de esa misma hora porque no conocemos el orden en que estuvo disponible. Un reemplazo asociado a una falla sí puede servir de historia en horas posteriores.

#### Información del catálogo: 2 variables conservadas

| Columna | Significado y origen | Valores posibles |
|---|---|---|
| `model` | Modelo de la máquina, unido desde el catálogo por `machineID`. | `model1`, `model2`, `model3` o `model4`; son categorías, no un orden de calidad. |
| `age` | Antigüedad en años registrada en el catálogo. Se conserva, no se recalcula cada hora. | Enteros no negativos; en este catálogo van de 0 a 20. Cero es válido. |

**Cuenta final:** 4 sensores originales + 16 variables móviles + 5 conteos de errores + 4 tiempos desde reemplazos + 2 datos del catálogo = **31 entradas**. De ellas, **25 son columnas nuevas de ingeniería de características** y 6 se conservan de las fuentes.

Además de esas entradas, `machineID` identifica la máquina y `datetime` identifica la hora. Son las dos claves usadas para integrar las tablas y ordenar el tiempo; no son etiquetas ni se agregan automáticamente al modelo como predictores.

In [3]:
# Esta función usa los cuatro archivos de entradas; no utiliza las fallas para crear predictores.
variables = crear_variables(datos)

# Ver únicamente ejemplos de entrenamiento. machineID y datetime son claves, no las respuestas.
ejemplos_entrenamiento = variables.loc[variables["datetime"] < PERIODOS["entrenamiento"][1]]
columnas_ejemplo = ["machineID", "datetime", "volt", "volt_media_3h", "volt_desv_3h",
                    "volt_media_24h", "error1_conteo_24h", "comp1_horas_desde_reemplazo", "model", "age"]
display(ejemplos_entrenamiento[columnas_ejemplo].head(6))

# Contar vacíos solo en entrenamiento para explicar los casos de historia insuficiente.
faltantes = ejemplos_entrenamiento.isna().sum()
display(faltantes[faltantes > 0].rename("faltantes_entrenamiento").to_frame())
print("Variables de entrada:", len(variables.columns) - 2)

,machineID,datetime,volt,volt_media_3h,volt_desv_3h,volt_media_24h,error1_conteo_24h,comp1_horas_desde_reemplazo,model,age
0,1,2015-01-01 06:00:00,176.217853,NaN,NaN,NaN,0,456.0,model3,18
1,1,2015-01-01 07:00:00,162.879223,NaN,NaN,NaN,0,457.0,model3,18
2,1,2015-01-01 08:00:00,170.989902,170.028993,6.721032,NaN,0,458.0,model3,18
3,1,2015-01-01 09:00:00,162.462833,165.443986,4.807415,NaN,0,459.0,model3,18
4,1,2015-01-01 10:00:00,157.610021,163.687586,6.773501,NaN,0,460.0,model3,18
5,1,2015-01-01 11:00:00,172.504839,164.192565,7.596570,NaN,0,461.0,model3,18


,faltantes_entrenamiento
volt_media_3h,200
volt_desv_3h,200
rotate_media_3h,200
rotate_desv_3h,200
pressure_media_3h,200
pressure_desv_3h,200
vibration_media_3h,200
vibration_desv_3h,200
volt_media_24h,2300
volt_desv_24h,2300


Variables de entrada: 31


Quedan **31 variables de entrada**, además de las dos claves. Los vacíos iniciales de los promedios y desviaciones son esperados: **200 por cada variable de 3 horas y 2,300 por cada variable de 24 horas**, equivalentes a 2 y 23 horas de las 100 máquinas.

Estos vacíos no se rellenan con observaciones futuras. En el notebook de modelos ajustaremos el tratamiento de faltantes y la codificación de `model` solo con entrenamiento. `machineID` y `datetime` permiten relacionar y ordenar filas; no se incluirán automáticamente como predictores.

## 4. Agregar las dos etiquetas y separar por fechas

- `etiqueta_binaria`: **1** si ocurre al menos una falla en **(t, t + 24 h]**; **0** si no ocurre ninguna.
- `etiqueta_multiclase`: **ninguno, comp1, comp2, comp3 o comp4**. Se elige el componente de la primera falla futura. Si varios fallan en esa misma primera hora, se desempata con el orden comp1–comp4. Es una convención reproducible, no una prioridad de gravedad.

La búsqueda se realiza por máquina y componente. Una falla en `t` queda fuera; una exactamente en `t + 24 h` sí entra. Las fallas sin mantenimiento coincidente se conservan como fallas.

| Conjunto | Periodo | Última hora conservada después del margen |
|---|---|---|
| Entrenamiento | Enero–agosto de 2015 | 30 de agosto, 23:00 |
| Validación | Septiembre–octubre de 2015 | 30 de octubre, 23:00 |
| Prueba | Noviembre–diciembre de 2015 | 30 de diciembre, 23:00 |

Conservamos solo horas con **t + 24 h < límite del conjunto**. Excluir las últimas 24 horas evita que entrenamiento dependa de septiembre, validación de noviembre o prueba de un seguimiento no garantizado en 2016. Esas horas se retiran, no se convierten en negativas.

### Cómo interpretar las dos columnas de respuesta

| Columna | Cómo se obtiene | Valores posibles |
|---|---|---|
| `etiqueta_binaria` | Consultar si existe alguna falla de esa máquina después de `t` y hasta 24 horas después. | `0`: ninguna; `1`: al menos una falla. No es una probabilidad predicha, sino la respuesta observada con la que aprenderá el modelo. |
| `etiqueta_multiclase` | Elegir el componente de la primera falla dentro del mismo horizonte, aplicando el desempate indicado. | Texto: `ninguno`, `comp1`, `comp2`, `comp3` o `comp4`. |

**Ejemplo ilustrativo:** si a las 10:00 sabemos retrospectivamente que comp2 falló a las 18:00 de ese día, las respuestas son `1` y `comp2`. Si no hubo fallas en las siguientes 24 horas, son `0` y `ninguno`. La información futura se usa para construir estas respuestas, nunca las variables de entrada.

Así, cada CSV termina con **35 columnas: 31 entradas + 2 claves + 2 etiquetas**.

In [4]:
# Agregar objetivos y aplicar los límites con la misma función para los tres conjuntos.
conjuntos, resumenes = preparar_conjuntos(variables, datos["fallas"])
display(resumenes["particion"])

# Mostrar las dos etiquetas junto a las mediciones, únicamente con ejemplos de entrenamiento.
entrenamiento = conjuntos["entrenamiento"]
ejemplos_positivos = entrenamiento.loc[entrenamiento["etiqueta_binaria"] == 1]
display(ejemplos_positivos[["machineID", "datetime", "volt", "model", "etiqueta_binaria", "etiqueta_multiclase"]].head(6))

,conjunto,filas,excluidas_frontera,primera_hora,ultima_hora
0,entrenamiento,580200,2400,2015-01-01 06:00:00,2015-08-30 23:00:00
1,validación,144000,2400,2015-09-01 00:00:00,2015-10-30 23:00:00
2,prueba,144000,2400,2015-11-01 00:00:00,2015-12-30 23:00:00


,machineID,datetime,volt,model,etiqueta_binaria,etiqueta_multiclase
72,1,2015-01-04 06:00:00,165.010140,model3,1,comp4
73,1,2015-01-04 07:00:00,142.666469,model3,1,comp4
74,1,2015-01-04 08:00:00,191.168936,model3,1,comp4
75,1,2015-01-04 09:00:00,157.436263,model3,1,comp4
76,1,2015-01-04 10:00:00,153.143558,model3,1,comp4
77,1,2015-01-04 11:00:00,215.656488,model3,1,comp4


Se conservan **580,200 filas de entrenamiento, 144,000 de validación y 144,000 de prueba**. Se excluyen 2,400 horas-máquina en cada frontera, equivalentes a 24 horas para 100 máquinas.

Cada fila ya tiene variables y respuestas juntas. La integración no multiplica registros. Los indicadores auxiliares usados para contar varios componentes se mantienen solo durante el cálculo de resúmenes: los CSV finales contienen únicamente las dos etiquetas principales.

## 5. Revisar desbalance y fallas simultáneas

Mostramos las proporciones de entrenamiento y validación. La frecuencia por componente cuenta todos los componentes del horizonte; puede sumar más que la clase única, que solo elige uno.

También documentamos cuántas horas tienen varios componentes o empate en la primera falla. No se muestran distribuciones ni ejemplos de etiquetas de prueba.

In [5]:
# Estos resúmenes se calculan dentro del script solo para entrenamiento y validación.
display(resumenes["binario"].round(3))
display(resumenes["clases"].round(3))
display(resumenes["componentes"].round(3))
display(resumenes["empates"])

,conjunto,filas,positivos,negativos,porcentaje_positivos
0,entrenamiento,580200,11742,568458,2.024
1,validación,144000,2652,141348,1.842


,conjunto,clase,horas,porcentaje
0,entrenamiento,ninguno,568458,97.976
1,entrenamiento,comp1,3357,0.579
2,entrenamiento,comp2,3849,0.663
3,entrenamiento,comp3,2001,0.345
4,entrenamiento,comp4,2535,0.437
5,validación,ninguno,141348,98.158
6,validación,comp1,498,0.346
7,validación,comp2,1080,0.750
8,validación,comp3,426,0.296
9,validación,comp4,648,0.450


,conjunto,componente,horas_positivas,porcentaje
0,entrenamiento,comp1,3357,0.579
1,entrenamiento,comp2,4017,0.692
2,entrenamiento,comp3,2163,0.373
3,entrenamiento,comp4,2919,0.503
4,validación,comp1,498,0.346
5,validación,comp2,1080,0.750
6,validación,comp3,474,0.329
7,validación,comp4,720,0.500


,conjunto,varios_componentes,empates_primera_falla
0,entrenamiento,714,693
1,validación,120,120


En entrenamiento hay **11,742 horas positivas (2.024%)** y en validación **2,652 (1.842%)**. Cerca del 98% de las horas no anticipan fallas; la exactitud por sí sola sería poco informativa.

Hay **714 horas de entrenamiento y 120 de validación** con varios componentes futuros. En **693 y 120**, respectivamente, la primera falla es simultánea. La regla de desempate pierde parte de esa información en la clase única y puede influir en las métricas por componente; por eso se explica y se mantiene fija.

Estas cantidades son horas etiquetadas, no fallas físicas independientes: una falla puede hacer positivas hasta 24 horas anteriores. No hemos cambiado las clases ni aplicado estrategias de desbalance.

## 6. Verificar la preparación

Contrastaremos ejemplos de entrenamiento y validación con cálculos directos: sensores móviles, conteos de error, reemplazos anteriores y etiquetas futuras. También comprobaremos las fronteras y la ausencia de claves duplicadas en las tres particiones. Las verificaciones de prueba son estructurales, sin evaluar modelos ni explorar sus respuestas.

In [6]:
# Comprobar la estructura de los tres conjuntos y que ninguna etiqueta cruce su frontera.
columnas_esperadas = list(variables.columns) + ["etiqueta_binaria", "etiqueta_multiclase"]
for nombre, tabla in conjuntos.items():
    assert list(tabla.columns) == columnas_esperadas
    assert not tabla.duplicated(["machineID", "datetime"]).any()
    assert (tabla["datetime"] + HORIZONTE < PERIODOS[nombre][1]).all()
    assert tabla[["etiqueta_binaria", "etiqueta_multiclase"]].notna().all().all()

# Elegir horas al principio de cada periodo, donde importa conservar la historia anterior.
verificaciones = []
for nombre in ["entrenamiento", "validación"]:
    tabla = conjuntos[nombre]
    muestra = pd.concat([tabla.head(3), tabla.tail(3), tabla.loc[tabla["etiqueta_binaria"] == 1].head(3)])
    muestra = muestra.drop_duplicates(["machineID", "datetime"])

    for fila in muestra.itertuples(index=False):
        maquina_id = fila.machineID
        t = fila.datetime

        for horas_ventana in [3, 24]:

            # Calcular directamente (t − L, t] a partir de la telemetría original filtrada.
            seleccion = ((datos["telemetria"]["machineID"] == maquina_id)
                         & (datos["telemetria"]["datetime"] > t - pd.Timedelta(hours=horas_ventana))
                         & (datos["telemetria"]["datetime"] <= t))
            historial = datos["telemetria"].loc[seleccion]
            for sensor in SENSORES:
                media = historial[sensor].mean() if len(historial) == horas_ventana else np.nan
                desv = historial[sensor].std(ddof=1) if len(historial) == horas_ventana else np.nan
                assert np.isclose(getattr(fila, f"{sensor}_media_{horas_ventana}h"), media, equal_nan=True)
                assert np.isclose(getattr(fila, f"{sensor}_desv_{horas_ventana}h"), desv, equal_nan=True)

        # Verificar cada conteo sin usar la agregación del pipeline.
        errores_previos = datos["errores"].loc[(datos["errores"]["machineID"] == maquina_id)
            & (datos["errores"]["datetime"] > t - HORIZONTE) & (datos["errores"]["datetime"] <= t)]
        for error in ERRORES:
            assert getattr(fila, error + "_conteo_24h") == int((errores_previos["errorID"] == error).sum())

        # Buscar el reemplazo más reciente estrictamente anterior, sin usar eventos futuros.
        for componente in COMPONENTES:
            anteriores = datos["mantenimiento"].loc[(datos["mantenimiento"]["machineID"] == maquina_id)
                & (datos["mantenimiento"]["comp"] == componente) & (datos["mantenimiento"]["datetime"] < t), "datetime"]
            esperado = (t - anteriores.max()).total_seconds() / 3600 if not anteriores.empty else np.nan
            assert np.isclose(getattr(fila, componente + "_horas_desde_reemplazo"), esperado, equal_nan=True)

        # Contrastar las dos etiquetas con una búsqueda directa en (t, t + 24h].
        futuras = datos["fallas"].loc[(datos["fallas"]["machineID"] == maquina_id)
            & (datos["fallas"]["datetime"] > t) & (datos["fallas"]["datetime"] <= t + HORIZONTE)]
        assert fila.etiqueta_binaria == int(not futuras.empty)
        clase = "ninguno" if futuras.empty else futuras.sort_values(["datetime", "failure"]).iloc[0]["failure"]
        assert fila.etiqueta_multiclase == clase

    verificaciones.append({"conjunto": nombre, "filas_contrastadas": len(muestra), "resultado": "Correcto"})

display(pd.DataFrame(verificaciones))

,conjunto,filas_contrastadas,resultado
0,entrenamiento,9,Correcto
1,validación,9,Correcto


Las comprobaciones contrastan los resultados con las fuentes, no solo entre columnas calculadas. Verifican que septiembre pueda utilizar historia de agosto y que las variables de reemplazos nunca busquen hacia adelante. Los vacíos de ventanas incompletas se conservan.

Esto no garantiza que los registros industriales siempre estuvieran cargados en tiempo real: en una implementación real habría que conocer también los retrasos de captura de sensores, errores y mantenimiento.

## 7. Guardar los tres conjuntos finales

Cada archivo contiene **35 columnas**: dos claves, 31 variables de entrada y dos etiquetas. No se guardan tablas intermedias de etiquetas ni resúmenes adicionales.

Los vacíos se representan como campos vacíos del CSV. Al leerlos, pandas los interpreta como valores faltantes. Los archivos originales de `Datos` no se modifican.

Los tres archivos generados se guardan en **`Datos/Datos_preparados/`**, separados de los cinco CSV originales que permanecen directamente en `Datos/`.

In [7]:
# Generar o reemplazar los tres CSV completos en Datos/Datos_preparados.
guardar_conjuntos(conjuntos, ruta_repo)

# Comprobar cantidad de filas y columnas leyendo por bloques, sin cargar otra copia completa.
for nombre, tabla in conjuntos.items():
    archivo = ruta_repo / "Datos" / "Datos_preparados" / (nombre + ".csv")
    filas_leidas = 0
    for bloque in pd.read_csv(archivo, chunksize=100000):
        assert list(bloque.columns) == list(tabla.columns)
        filas_leidas += len(bloque)
    assert filas_leidas == len(tabla)

print("Los tres CSV se guardaron y su estructura se verificó.")

Guardado: entrenamiento.csv
Guardado: validación.csv
Guardado: prueba.csv
Los tres CSV se guardaron y su estructura se verificó.


## 8. Reproducir toda la preparación con el pipeline

**`preparar_datos.py` es el pipeline de datos completo.** Un pipeline es simplemente una secuencia de pasos que se ejecuta en orden para transformar los archivos originales en los conjuntos listos para el siguiente paso de modelado.

Desde la raíz del repositorio, este comando ejecuta toda esa preparación:

```bash
python Preprocesamiento/preparar_datos.py
```

El pipeline hace lo siguiente, sin tener que correr manualmente las celdas del notebook:

1. **Lee los cinco CSV originales** de `Datos` y comprueba sus claves.
2. **Crea e integra las variables de entrada:** medias y desviaciones de 3 y 24 horas, conteos de errores de 24 horas, tiempo desde reemplazos, sensores originales, modelo y antigüedad. Los cálculos respetan el historial de cada máquina.
3. **Aplica la partición temporal:** entrenamiento de enero a agosto, validación de septiembre a octubre y prueba de noviembre a diciembre. Excluye las horas cuyo horizonte de 24 horas cruza la frontera del conjunto.
4. **Crea las etiquetas binaria y multiclase** de cada partición con sus propios eventos de falla, sin pasar información de otro periodo a sus respuestas.
5. **Guarda los tres conjuntos completos en `Datos/Datos_preparados`:** `entrenamiento.csv`, `validación.csv` y `prueba.csv`, con las variables y las dos etiquetas en cada fila. Al repetir el comando, regenera esos archivos.

El orden es importante: las variables históricas se calculan antes de separar periodos para que septiembre pueda aprovechar la historia de agosto. Cada cálculo usa únicamente información disponible hasta su hora; no mira hacia adelante. Las etiquetas sí consultan las siguientes 24 horas, porque representan lo que queremos predecir.

**El notebook y el pipeline utilizan las mismas funciones.** El notebook permite entender cada paso y revisar tablas; el script permite repetir la preparación completa con un comando. No es un procesamiento adicional ni una segunda versión del código.

**Siguiente paso:** modelos de línea base con regresión logística y boosting. Se ajustarán con entrenamiento y se compararán en validación. La imputación de faltantes, las codificaciones y cualquier transformación que aprenda parámetros se ajustarán solo con entrenamiento. Prueba queda reservada para la evaluación final en la semana 3, sin orientar esas decisiones.